# Natera (NTER) Options Hedging Analysis

## Portfolio Overview
- **Current Position**: 3,000 shares of Natera (NTER)
- **Analysis Date**: November 24, 2024
- **Objective**: Evaluate and recommend optimal hedging strategies to protect downside risk while maintaining upside potential

This notebook will analyze several hedging strategies:
1. **Protective Puts** - Buying put options to protect against downside
2. **Covered Calls** - Selling call options to generate income
3. **Collar Strategy** - Combining protective puts with covered calls
4. **Put Spreads** - Using put spreads to reduce hedging costs

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import yfinance as yf
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Current Position Analysis

First, let's fetch the current stock data for Natera and analyze your position.

In [ ]:
# Portfolio Parameters
SHARES_OWNED = 3000
TICKER = "NTER"  # Natera ticker symbol

# Fetch current stock data
stock = yf.Ticker(TICKER)

# Get current stock price and info
try:
    current_price = stock.info['currentPrice']
except:
    # Fallback to regularMarketPrice if currentPrice not available
    current_price = stock.info.get('regularMarketPrice', stock.info.get('previousClose', 0))

# Get stock info
stock_info = {
    'Symbol': TICKER,
    'Company Name': stock.info.get('longName', 'Natera Inc.'),
    'Current Price': current_price,
    'Shares Owned': SHARES_OWNED,
    'Position Value': current_price * SHARES_OWNED,
    '52 Week High': stock.info.get('fiftyTwoWeekHigh', 'N/A'),
    '52 Week Low': stock.info.get('fiftyTwoWeekLow', 'N/A'),
    'Market Cap': stock.info.get('marketCap', 'N/A'),
    'Beta': stock.info.get('beta', 'N/A')
}

# Display position summary
print("=" * 60)
print("NATERA POSITION SUMMARY")
print("=" * 60)
for key, value in stock_info.items():
    if key == 'Position Value' and isinstance(value, (int, float)):
        print(f"{key}: ${value:,.2f}")
    elif key in ['Current Price', '52 Week High', '52 Week Low'] and isinstance(value, (int, float)):
        print(f"{key}: ${value:.2f}")
    elif key == 'Market Cap' and isinstance(value, (int, float)):
        print(f"{key}: ${value/1e9:.2f}B")
    else:
        print(f"{key}: {value}")

## 2. Historical Price and Volatility Analysis

Understanding the stock's historical volatility is crucial for evaluating options strategies.

In [ ]:
# Fetch historical data (1 year)
end_date = datetime.now()
start_date = end_date - timedelta(days=365)
hist_data = stock.history(start=start_date, end=end_date)

# Calculate returns and volatility
hist_data['Daily_Return'] = hist_data['Close'].pct_change()
hist_data['Log_Return'] = np.log(hist_data['Close'] / hist_data['Close'].shift(1))

# Calculate volatility metrics
daily_volatility = hist_data['Daily_Return'].std()
annual_volatility = daily_volatility * np.sqrt(252)  # Annualized volatility

# Calculate different period volatilities
volatility_30d = hist_data['Daily_Return'].tail(30).std() * np.sqrt(252)
volatility_60d = hist_data['Daily_Return'].tail(60).std() * np.sqrt(252)
volatility_90d = hist_data['Daily_Return'].tail(90).std() * np.sqrt(252)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Price History
axes[0, 0].plot(hist_data.index, hist_data['Close'], linewidth=2)
axes[0, 0].axhline(y=current_price, color='r', linestyle='--', label=f'Current: ${current_price:.2f}')
axes[0, 0].set_title('NTER Stock Price - Last 12 Months', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Daily Returns Distribution
axes[0, 1].hist(hist_data['Daily_Return'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Daily Return (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Rolling 30-day Volatility
rolling_vol = hist_data['Daily_Return'].rolling(window=30).std() * np.sqrt(252)
axes[1, 0].plot(rolling_vol.index, rolling_vol, linewidth=2, color='orange')
axes[1, 0].axhline(y=annual_volatility, color='r', linestyle='--', label=f'Annual Avg: {annual_volatility:.1%}')
axes[1, 0].set_title('30-Day Rolling Volatility (Annualized)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Volatility')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Price with Moving Averages
axes[1, 1].plot(hist_data.index, hist_data['Close'], label='Price', linewidth=2)
axes[1, 1].plot(hist_data.index, hist_data['Close'].rolling(20).mean(), label='20-day MA', alpha=0.7)
axes[1, 1].plot(hist_data.index, hist_data['Close'].rolling(50).mean(), label='50-day MA', alpha=0.7)
axes[1, 1].set_title('Price with Moving Averages', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print volatility summary
print("\n" + "=" * 60)
print("VOLATILITY ANALYSIS")
print("=" * 60)
print(f"Annual Historical Volatility: {annual_volatility:.1%}")
print(f"30-Day Volatility: {volatility_30d:.1%}")
print(f"60-Day Volatility: {volatility_60d:.1%}")
print(f"90-Day Volatility: {volatility_90d:.1%}")
print(f"\nDaily Value at Risk (95% confidence): ${current_price * 1.65 * daily_volatility * SHARES_OWNED:,.0f}")
print(f"Monthly Value at Risk (95% confidence): ${current_price * 1.65 * daily_volatility * np.sqrt(21) * SHARES_OWNED:,.0f}")

## 3. Available Options Analysis

Let's examine the available options chains and identify suitable contracts for hedging.

In [ ]:
# Get available expiration dates
expirations = stock.options
print(f"Available expiration dates: {len(expirations)}")
print(f"Next 5 expirations: {expirations[:5] if len(expirations) >= 5 else expirations}")

# Select expiration dates for analysis (30, 60, 90 days out approximately)
target_dates = []
today = datetime.now()

for exp in expirations[:8]:  # Check first 8 expirations
    exp_date = datetime.strptime(exp, '%Y-%m-%d')
    days_to_exp = (exp_date - today).days
    if 20 <= days_to_exp <= 120:
        target_dates.append({
            'expiration': exp,
            'days_to_expiration': days_to_exp,
            'exp_date': exp_date
        })

print(f"\nSelected expiration dates for analysis:")
for td in target_dates[:4]:  # Analyze up to 4 expiration dates
    print(f"  - {td['expiration']}: {td['days_to_expiration']} days")

# Analyze options chains for selected dates
options_data = []

for td in target_dates[:3]:  # Analyze first 3 suitable expirations
    exp = td['expiration']
    days_to_exp = td['days_to_expiration']
    
    # Get options chain
    opt_chain = stock.option_chain(exp)
    
    # Analyze Puts for hedging
    puts = opt_chain.puts
    puts['moneyness'] = puts['strike'] / current_price
    puts['days_to_exp'] = days_to_exp
    
    # Analyze Calls for covered calls
    calls = opt_chain.calls
    calls['moneyness'] = calls['strike'] / current_price
    calls['days_to_exp'] = days_to_exp
    
    options_data.append({
        'expiration': exp,
        'days_to_exp': days_to_exp,
        'puts': puts,
        'calls': calls
    })

## 4. Hedging Strategy Analysis

Now let's analyze different hedging strategies for your 3,000 shares.

In [ ]:
# Define hedging strategy analyzer
class HedgingStrategyAnalyzer:
    def __init__(self, current_price, shares, volatility):
        self.current_price = current_price
        self.shares = shares
        self.volatility = volatility
        self.position_value = current_price * shares
        
    def analyze_protective_put(self, put_strike, put_premium, days_to_exp):
        """Analyze protective put strategy"""
        # Calculate costs
        total_cost = put_premium * self.shares / 100  # Convert to total cost
        
        # Calculate breakeven
        breakeven = self.current_price + put_premium
        
        # Calculate protection levels
        max_loss = ((self.current_price - put_strike) + put_premium) * self.shares
        protection_percentage = (self.current_price - put_strike) / self.current_price
        
        return {
            'strategy': 'Protective Put',
            'strike': put_strike,
            'premium_per_share': put_premium,
            'total_cost': total_cost,
            'breakeven': breakeven,
            'max_loss': max_loss,
            'protection_level': protection_percentage,
            'days_to_exp': days_to_exp,
            'annual_cost': (total_cost / days_to_exp * 365) / self.position_value
        }
    
    def analyze_covered_call(self, call_strike, call_premium, days_to_exp):
        """Analyze covered call strategy"""
        # Calculate income
        total_income = call_premium * self.shares / 100
        
        # Calculate outcomes
        max_gain = ((call_strike - self.current_price) + call_premium) * self.shares
        breakeven = self.current_price - call_premium
        
        return {
            'strategy': 'Covered Call',
            'strike': call_strike,
            'premium_per_share': call_premium,
            'total_income': total_income,
            'breakeven': breakeven,
            'max_gain': max_gain,
            'cap_level': (call_strike - self.current_price) / self.current_price,
            'days_to_exp': days_to_exp,
            'annual_yield': (total_income / days_to_exp * 365) / self.position_value
        }
    
    def analyze_collar(self, put_strike, put_premium, call_strike, call_premium, days_to_exp):
        """Analyze collar strategy (protective put + covered call)"""
        net_cost = (put_premium - call_premium) * self.shares / 100
        
        max_loss = ((self.current_price - put_strike) + (put_premium - call_premium)) * self.shares
        max_gain = ((call_strike - self.current_price) - (put_premium - call_premium)) * self.shares
        
        return {
            'strategy': 'Collar',
            'put_strike': put_strike,
            'call_strike': call_strike,
            'net_cost_per_share': put_premium - call_premium,
            'total_net_cost': net_cost,
            'max_loss': max_loss,
            'max_gain': max_gain,
            'protection_level': (self.current_price - put_strike) / self.current_price,
            'cap_level': (call_strike - self.current_price) / self.current_price,
            'days_to_exp': days_to_exp
        }
    
    def calculate_pl_scenarios(self, strategy_type, params, price_range):
        """Calculate P&L for different price scenarios"""
        pl_scenarios = []
        
        for future_price in price_range:
            if strategy_type == 'protective_put':
                stock_pl = (future_price - self.current_price) * self.shares
                put_value = max(params['strike'] - future_price, 0) * self.shares
                total_pl = stock_pl + put_value - params['total_cost']
                
            elif strategy_type == 'covered_call':
                stock_pl = min(future_price - self.current_price, params['strike'] - self.current_price) * self.shares
                total_pl = stock_pl + params['total_income']
                
            elif strategy_type == 'collar':
                stock_pl = (future_price - self.current_price) * self.shares
                put_value = max(params['put_strike'] - future_price, 0) * self.shares
                call_obligation = -max(future_price - params['call_strike'], 0) * self.shares
                total_pl = stock_pl + put_value + call_obligation - params['total_net_cost']
                
            else:  # unhedged
                total_pl = (future_price - self.current_price) * self.shares
                
            pl_scenarios.append({
                'price': future_price,
                'pl': total_pl,
                'return': total_pl / self.position_value
            })
        
        return pd.DataFrame(pl_scenarios)

# Initialize analyzer
analyzer = HedgingStrategyAnalyzer(current_price, SHARES_OWNED, annual_volatility)

print("Hedging Strategy Analyzer initialized.")
print(f"Position Value: ${analyzer.position_value:,.2f}")
print(f"Analyzing strategies for {SHARES_OWNED} shares at ${current_price:.2f}/share")

### 4.1 Strategy 1: Protective Put Analysis

In [ ]:
# Analyze Protective Put Strategies
protective_put_strategies = []

if len(options_data) > 0:
    for opt_data in options_data[:2]:  # Analyze first 2 expirations
        puts = opt_data['puts']
        exp = opt_data['expiration']
        days = opt_data['days_to_exp']
        
        # Filter for relevant strikes (85% to 95% of current price)
        relevant_puts = puts[(puts['moneyness'] >= 0.85) & (puts['moneyness'] <= 0.95)]
        relevant_puts = relevant_puts.sort_values('strike', ascending=False)
        
        print(f"\n📊 Protective Put Options - Expiration: {exp} ({days} days)")
        print("-" * 80)
        
        for _, put in relevant_puts.head(3).iterrows():
            if pd.notna(put['lastPrice']) and put['lastPrice'] > 0:
                analysis = analyzer.analyze_protective_put(
                    put['strike'], 
                    put['lastPrice'],
                    days
                )
                protective_put_strategies.append(analysis)
                
                print(f"Strike: ${put['strike']:.2f} ({put['moneyness']:.1%} of current)")
                print(f"  Premium: ${put['lastPrice']:.2f}/share")
                print(f"  Total Cost: ${analysis['total_cost']:,.2f}")
                print(f"  Max Loss: ${analysis['max_loss']:,.2f}")
                print(f"  Protection: {analysis['protection_level']:.1%} downside protection")
                print(f"  Annual Cost: {analysis['annual_cost']:.2%} of position")
                print(f"  Bid/Ask: ${put['bid']:.2f} / ${put['ask']:.2f}")
                print(f"  Volume: {put['volume']:.0f}, Open Interest: {put['openInterest']:.0f}")
                print()

# Convert to DataFrame for easier analysis
if protective_put_strategies:
    pp_df = pd.DataFrame(protective_put_strategies)
    
    print("\n" + "=" * 60)
    print("RECOMMENDED PROTECTIVE PUT STRATEGIES")
    print("=" * 60)
    
    # Sort by annual cost and display top options
    pp_df_sorted = pp_df.sort_values('annual_cost').head(3)
    
    for idx, row in pp_df_sorted.iterrows():
        print(f"\nOption {idx+1}:")
        print(f"  Strike: ${row['strike']:.2f}")
        print(f"  Days to Expiration: {row['days_to_exp']}")
        print(f"  Cost: ${row['total_cost']:,.2f} ({row['annual_cost']:.1%} annualized)")
        print(f"  Protection: {row['protection_level']:.1%} downside")
        print(f"  Max Loss: ${row['max_loss']:,.2f}")
else:
    print("No protective put options data available. Using simulated data for demonstration...")
    # Create simulated protective put strategies for demonstration
    protective_put_strategies = [
        analyzer.analyze_protective_put(current_price * 0.90, current_price * 0.03, 30),
        analyzer.analyze_protective_put(current_price * 0.95, current_price * 0.05, 60),
        analyzer.analyze_protective_put(current_price * 0.85, current_price * 0.02, 90)
    ]
    pp_df = pd.DataFrame(protective_put_strategies)

### 4.2 Strategy 2: Covered Call Analysis

In [ ]:
# Analyze Covered Call Strategies
covered_call_strategies = []

if len(options_data) > 0:
    for opt_data in options_data[:2]:  # Analyze first 2 expirations
        calls = opt_data['calls']
        exp = opt_data['expiration']
        days = opt_data['days_to_exp']
        
        # Filter for relevant strikes (105% to 115% of current price)
        relevant_calls = calls[(calls['moneyness'] >= 1.05) & (calls['moneyness'] <= 1.15)]
        relevant_calls = relevant_calls.sort_values('strike')
        
        print(f"\n📊 Covered Call Options - Expiration: {exp} ({days} days)")
        print("-" * 80)
        
        for _, call in relevant_calls.head(3).iterrows():
            if pd.notna(call['lastPrice']) and call['lastPrice'] > 0:
                analysis = analyzer.analyze_covered_call(
                    call['strike'],
                    call['lastPrice'],
                    days
                )
                covered_call_strategies.append(analysis)
                
                print(f"Strike: ${call['strike']:.2f} ({call['moneyness']:.1%} of current)")
                print(f"  Premium: ${call['lastPrice']:.2f}/share")
                print(f"  Total Income: ${analysis['total_income']:,.2f}")
                print(f"  Max Gain: ${analysis['max_gain']:,.2f}")
                print(f"  Cap Level: {analysis['cap_level']:.1%} upside cap")
                print(f"  Annual Yield: {analysis['annual_yield']:.2%}")
                print(f"  Bid/Ask: ${call['bid']:.2f} / ${call['ask']:.2f}")
                print(f"  Volume: {call['volume']:.0f}, Open Interest: {call['openInterest']:.0f}")
                print()

# Convert to DataFrame for easier analysis
if covered_call_strategies:
    cc_df = pd.DataFrame(covered_call_strategies)
    
    print("\n" + "=" * 60)
    print("RECOMMENDED COVERED CALL STRATEGIES")
    print("=" * 60)
    
    # Sort by annual yield and display top options
    cc_df_sorted = cc_df.sort_values('annual_yield', ascending=False).head(3)
    
    for idx, row in cc_df_sorted.iterrows():
        print(f"\nOption {idx+1}:")
        print(f"  Strike: ${row['strike']:.2f}")
        print(f"  Days to Expiration: {row['days_to_exp']}")
        print(f"  Income: ${row['total_income']:,.2f} ({row['annual_yield']:.1%} annualized)")
        print(f"  Max Gain: ${row['max_gain']:,.2f}")
        print(f"  Upside Cap: {row['cap_level']:.1%}")
else:
    print("No covered call options data available. Using simulated data for demonstration...")
    # Create simulated covered call strategies for demonstration
    covered_call_strategies = [
        analyzer.analyze_covered_call(current_price * 1.10, current_price * 0.03, 30),
        analyzer.analyze_covered_call(current_price * 1.15, current_price * 0.02, 45),
        analyzer.analyze_covered_call(current_price * 1.05, current_price * 0.04, 60)
    ]
    cc_df = pd.DataFrame(covered_call_strategies)

### 4.3 Strategy 3: Collar Strategy Analysis

In [ ]:
# Analyze Collar Strategies (Protective Put + Covered Call)
collar_strategies = []

if len(options_data) > 0:
    for opt_data in options_data[:2]:  # Analyze first 2 expirations
        puts = opt_data['puts']
        calls = opt_data['calls']
        exp = opt_data['expiration']
        days = opt_data['days_to_exp']
        
        print(f"\n📊 Collar Strategy - Expiration: {exp} ({days} days)")
        print("-" * 80)
        
        # Find optimal collar combinations
        # Typical collar: 90-95% put, 105-110% call
        put_strikes = [current_price * 0.90, current_price * 0.95]
        call_strikes = [current_price * 1.05, current_price * 1.10]
        
        for put_strike in put_strikes:
            # Find closest put
            put_row = puts.iloc[(puts['strike'] - put_strike).abs().argsort()[:1]]
            if len(put_row) > 0:
                put = put_row.iloc[0]
                
                for call_strike in call_strikes:
                    # Find closest call
                    call_row = calls.iloc[(calls['strike'] - call_strike).abs().argsort()[:1]]
                    if len(call_row) > 0:
                        call = call_row.iloc[0]
                        
                        if pd.notna(put['lastPrice']) and pd.notna(call['lastPrice']):
                            analysis = analyzer.analyze_collar(
                                put['strike'],
                                put['lastPrice'],
                                call['strike'],
                                call['lastPrice'],
                                days
                            )
                            collar_strategies.append(analysis)
                            
                            print(f"Collar: Put ${put['strike']:.2f} / Call ${call['strike']:.2f}")
                            print(f"  Net Cost: ${analysis['net_cost_per_share']:.2f}/share")
                            print(f"  Total Net Cost: ${analysis['total_net_cost']:,.2f}")
                            print(f"  Max Loss: ${analysis['max_loss']:,.2f} ({analysis['protection_level']:.1%})")
                            print(f"  Max Gain: ${analysis['max_gain']:,.2f} ({analysis['cap_level']:.1%})")
                            print()

# Convert to DataFrame for easier analysis
if collar_strategies:
    collar_df = pd.DataFrame(collar_strategies)
    
    print("\n" + "=" * 60)
    print("RECOMMENDED COLLAR STRATEGIES")
    print("=" * 60)
    
    # Sort by net cost and display top options
    collar_df_sorted = collar_df.sort_values('total_net_cost').head(3)
    
    for idx, row in collar_df_sorted.iterrows():
        print(f"\nOption {idx+1}:")
        print(f"  Put Strike: ${row['put_strike']:.2f} / Call Strike: ${row['call_strike']:.2f}")
        print(f"  Days to Expiration: {row['days_to_exp']}")
        print(f"  Net Cost: ${row['total_net_cost']:,.2f}")
        print(f"  Protection: {row['protection_level']:.1%} downside")
        print(f"  Cap: {row['cap_level']:.1%} upside")
        print(f"  Max Loss: ${row['max_loss']:,.2f}")
        print(f"  Max Gain: ${row['max_gain']:,.2f}")
else:
    print("No collar options data available. Using simulated data for demonstration...")
    # Create simulated collar strategies for demonstration
    collar_strategies = [
        analyzer.analyze_collar(current_price * 0.90, current_price * 0.03, 
                               current_price * 1.10, current_price * 0.025, 45),
        analyzer.analyze_collar(current_price * 0.95, current_price * 0.04,
                               current_price * 1.05, current_price * 0.035, 60),
        analyzer.analyze_collar(current_price * 0.92, current_price * 0.035,
                               current_price * 1.08, current_price * 0.03, 90)
    ]
    collar_df = pd.DataFrame(collar_strategies)

## 5. Profit/Loss Scenario Analysis

Let's visualize how each strategy performs under different price scenarios.

In [ ]:
# Create price scenarios (from -30% to +30%)
price_range = np.linspace(current_price * 0.7, current_price * 1.3, 100)

# Select best strategy from each type for comparison
best_pp = pp_df.iloc[0] if len(pp_df) > 0 else protective_put_strategies[0]
best_cc = cc_df.iloc[0] if len(cc_df) > 0 else covered_call_strategies[0]
best_collar = collar_df.iloc[0] if len(collar_df) > 0 else collar_strategies[0]

# Calculate P&L for each strategy
unhedged_pl = analyzer.calculate_pl_scenarios('unhedged', {}, price_range)
pp_pl = analyzer.calculate_pl_scenarios('protective_put', best_pp, price_range)
cc_pl = analyzer.calculate_pl_scenarios('covered_call', best_cc, price_range)
collar_pl = analyzer.calculate_pl_scenarios('collar', best_collar, price_range)

# Create comprehensive P&L visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Dollar P&L Comparison
axes[0, 0].plot(price_range, unhedged_pl['pl'], label='Unhedged', linewidth=2, linestyle='--')
axes[0, 0].plot(price_range, pp_pl['pl'], label='Protective Put', linewidth=2)
axes[0, 0].plot(price_range, cc_pl['pl'], label='Covered Call', linewidth=2)
axes[0, 0].plot(price_range, collar_pl['pl'], label='Collar', linewidth=2)
axes[0, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[0, 0].axvline(x=current_price, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Profit/Loss Comparison ($)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Stock Price at Expiration')
axes[0, 0].set_ylabel('Profit/Loss ($)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Percentage Return Comparison
axes[0, 1].plot(price_range, unhedged_pl['return']*100, label='Unhedged', linewidth=2, linestyle='--')
axes[0, 1].plot(price_range, pp_pl['return']*100, label='Protective Put', linewidth=2)
axes[0, 1].plot(price_range, cc_pl['return']*100, label='Covered Call', linewidth=2)
axes[0, 1].plot(price_range, collar_pl['return']*100, label='Collar', linewidth=2)
axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[0, 1].axvline(x=current_price, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Return Comparison (%)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Stock Price at Expiration')
axes[0, 1].set_ylabel('Return (%)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Downside Protection Focus
downside_range = np.linspace(current_price * 0.7, current_price, 50)
unhedged_down = analyzer.calculate_pl_scenarios('unhedged', {}, downside_range)
pp_down = analyzer.calculate_pl_scenarios('protective_put', best_pp, downside_range)
collar_down = analyzer.calculate_pl_scenarios('collar', best_collar, downside_range)

axes[1, 0].plot(downside_range, unhedged_down['pl'], label='Unhedged', linewidth=2, linestyle='--')
axes[1, 0].plot(downside_range, pp_down['pl'], label='Protective Put', linewidth=2)
axes[1, 0].plot(downside_range, collar_down['pl'], label='Collar', linewidth=2)
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1, 0].set_title('Downside Protection Analysis', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Stock Price at Expiration')
axes[1, 0].set_ylabel('Profit/Loss ($)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Strategy Summary Table
axes[1, 1].axis('off')
summary_data = []

# Unhedged summary
summary_data.append(['Unhedged', 'N/A', '$0', 'Unlimited', f'-${current_price * SHARES_OWNED:,.0f}'])

# Protective Put summary
pp_cost = f"${best_pp['total_cost']:,.0f}" if 'total_cost' in best_pp else 'N/A'
pp_max_loss = f"${best_pp['max_loss']:,.0f}" if 'max_loss' in best_pp else 'N/A'
summary_data.append(['Protective Put', f"${best_pp['strike']:.0f}", pp_cost, 'Unlimited', pp_max_loss])

# Covered Call summary
cc_income = f"${best_cc['total_income']:,.0f}" if 'total_income' in best_cc else 'N/A'
cc_max_gain = f"${best_cc['max_gain']:,.0f}" if 'max_gain' in best_cc else 'N/A'
summary_data.append(['Covered Call', f"${best_cc['strike']:.0f}", f"-{cc_income}", cc_max_gain, 'Unlimited'])

# Collar summary
collar_cost = f"${best_collar['total_net_cost']:,.0f}" if 'total_net_cost' in best_collar else 'N/A'
collar_max_gain = f"${best_collar['max_gain']:,.0f}" if 'max_gain' in best_collar else 'N/A'
collar_max_loss = f"${best_collar['max_loss']:,.0f}" if 'max_loss' in best_collar else 'N/A'
summary_data.append(['Collar', f"P${best_collar['put_strike']:.0f}/C${best_collar['call_strike']:.0f}", 
                    collar_cost, collar_max_gain, collar_max_loss])

table = axes[1, 1].table(cellText=summary_data,
                        colLabels=['Strategy', 'Strike(s)', 'Net Cost', 'Max Gain', 'Max Loss'],
                        cellLoc='center',
                        loc='center',
                        colWidths=[0.25, 0.25, 0.2, 0.15, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# Style the table
for i in range(len(summary_data) + 1):
    for j in range(5):
        if i == 0:
            table[(i, j)].set_facecolor('#40466e')
            table[(i, j)].set_text_props(weight='bold', color='white')
        else:
            table[(i, j)].set_facecolor('#f1f1f2' if i % 2 == 0 else 'white')

axes[1, 1].set_title('Strategy Comparison Summary', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("KEY INSIGHTS FROM P&L ANALYSIS")
print("=" * 80)
print(f"At current price of ${current_price:.2f}:")
print(f"• Unhedged position has unlimited upside but full downside exposure")
print(f"• Protective Put limits downside to ${best_pp['max_loss']:,.0f} while keeping upside")
print(f"• Covered Call generates ${best_cc['total_income']:,.0f} income but caps upside at ${best_cc['strike']:.0f}")
print(f"• Collar provides balanced protection with limited cost")

## 6. Risk Metrics and Greeks Analysis

In [ ]:
# Calculate Risk Metrics for each strategy
def calculate_risk_metrics(strategy_pl_df, strategy_name):
    """Calculate various risk metrics for a strategy"""
    
    # Find key points
    current_idx = np.abs(strategy_pl_df['price'] - current_price).argmin()
    current_pl = strategy_pl_df.iloc[current_idx]['pl']
    
    # Calculate metrics at various probability levels
    # Assuming normal distribution of returns
    one_std_down = current_price * (1 - annual_volatility / np.sqrt(252/30))  # 30-day 1 std move
    two_std_down = current_price * (1 - 2 * annual_volatility / np.sqrt(252/30))
    one_std_up = current_price * (1 + annual_volatility / np.sqrt(252/30))
    
    # Find P&L at these levels
    pl_1std_down = strategy_pl_df.iloc[np.abs(strategy_pl_df['price'] - one_std_down).argmin()]['pl']
    pl_2std_down = strategy_pl_df.iloc[np.abs(strategy_pl_df['price'] - two_std_down).argmin()]['pl']
    pl_1std_up = strategy_pl_df.iloc[np.abs(strategy_pl_df['price'] - one_std_up).argmin()]['pl']
    
    # Calculate Value at Risk (VaR) - 95% confidence
    var_95 = pl_2std_down  # Approximately 95% confidence
    
    # Calculate maximum drawdown
    max_loss = strategy_pl_df['pl'].min()
    max_gain = strategy_pl_df['pl'].max()
    
    return {
        'Strategy': strategy_name,
        'Max Loss': max_loss,
        'Max Gain': max_gain,
        'VaR (95%)': var_95,
        'P&L @ -1σ': pl_1std_down,
        'P&L @ +1σ': pl_1std_up,
        'P&L @ -2σ': pl_2std_down,
        'Breakeven': strategy_pl_df[strategy_pl_df['pl'] >= 0]['price'].min() if any(strategy_pl_df['pl'] >= 0) else None
    }

# Calculate risk metrics for all strategies
risk_metrics = []
risk_metrics.append(calculate_risk_metrics(unhedged_pl, 'Unhedged'))
risk_metrics.append(calculate_risk_metrics(pp_pl, 'Protective Put'))
risk_metrics.append(calculate_risk_metrics(cc_pl, 'Covered Call'))
risk_metrics.append(calculate_risk_metrics(collar_pl, 'Collar'))

risk_df = pd.DataFrame(risk_metrics)

# Display risk metrics
print("=" * 80)
print("RISK METRICS COMPARISON")
print("=" * 80)
print("\nExpected P&L at Different Probability Levels:")
print(risk_df.to_string(index=False))

# Create probability distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Probability of Different Outcomes
prob_levels = ['-2σ', '-1σ', 'Current', '+1σ', '+2σ']
prob_prices = [
    current_price * (1 - 2 * annual_volatility / np.sqrt(252/30)),
    current_price * (1 - annual_volatility / np.sqrt(252/30)),
    current_price,
    current_price * (1 + annual_volatility / np.sqrt(252/30)),
    current_price * (1 + 2 * annual_volatility / np.sqrt(252/30))
]

strategies = ['Unhedged', 'Protective Put', 'Covered Call', 'Collar']
x = np.arange(len(prob_levels))
width = 0.2

for i, strat_pl in enumerate([unhedged_pl, pp_pl, cc_pl, collar_pl]):
    pls = []
    for price in prob_prices:
        pl = strat_pl.iloc[np.abs(strat_pl['price'] - price).argmin()]['pl']
        pls.append(pl)
    axes[0].bar(x + i * width, pls, width, label=strategies[i])

axes[0].set_xlabel('Price Movement Scenario')
axes[0].set_ylabel('Profit/Loss ($)')
axes[0].set_title('P&L at Different Probability Levels (30-day horizon)', fontsize=14, fontweight='bold')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(prob_levels)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=1)

# Plot 2: Risk-Return Profile
returns = []
risks = []
labels = []

for i, (strat_pl, name) in enumerate([(unhedged_pl, 'Unhedged'), 
                                       (pp_pl, 'Protective Put'),
                                       (cc_pl, 'Covered Call'),
                                       (collar_pl, 'Collar')]):
    # Calculate expected return (using probability-weighted outcomes)
    exp_return = (strat_pl.iloc[np.abs(strat_pl['price'] - prob_prices[3]).argmin()]['return'] * 0.34 +  # +1σ
                 strat_pl.iloc[np.abs(strat_pl['price'] - current_price).argmin()]['return'] * 0.32 +  # Current
                 strat_pl.iloc[np.abs(strat_pl['price'] - prob_prices[1]).argmin()]['return'] * 0.34)  # -1σ
    
    # Calculate risk (standard deviation of returns)
    risk = np.std([strat_pl.iloc[np.abs(strat_pl['price'] - p).argmin()]['return'] for p in prob_prices])
    
    returns.append(exp_return * 100)
    risks.append(risk * 100)
    labels.append(name)

axes[1].scatter(risks, returns, s=200, alpha=0.6)
for i, label in enumerate(labels):
    axes[1].annotate(label, (risks[i], returns[i]), fontsize=10, ha='center')

axes[1].set_xlabel('Risk (Standard Deviation %)')
axes[1].set_ylabel('Expected Return (%)')
axes[1].set_title('Risk-Return Profile of Strategies', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=1)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("PROBABILITY ANALYSIS (30-day horizon)")
print("=" * 80)
print(f"Based on {annual_volatility:.1%} annual volatility:")
print(f"• 68% probability: Stock between ${prob_prices[1]:.2f} and ${prob_prices[3]:.2f}")
print(f"• 95% probability: Stock between ${prob_prices[0]:.2f} and ${prob_prices[4]:.2f}")
print(f"• Current price: ${current_price:.2f}")

## 7. Final Recommendations

In [ ]:
# Generate tailored recommendations based on different investor profiles
print("=" * 80)
print("PERSONALIZED HEDGING RECOMMENDATIONS")
print("=" * 80)

recommendations = {
    'Conservative': {
        'primary': 'Protective Put',
        'details': f"""
        ✅ RECOMMENDED: Protective Put Strategy
        
        Strike: ${best_pp['strike']:.2f} (90-95% of current price)
        Expiration: {best_pp['days_to_exp']} days
        Cost: ${best_pp['total_cost']:,.2f}
        
        Benefits:
        • Limits maximum loss to ${best_pp['max_loss']:,.2f}
        • Maintains all upside potential
        • Peace of mind during volatile periods
        • Can be rolled forward as needed
        
        Ideal for: Investors who want strong downside protection and 
        are willing to pay premium for insurance.
        """
    },
    'Income-Focused': {
        'primary': 'Covered Call',
        'details': f"""
        ✅ RECOMMENDED: Covered Call Strategy
        
        Strike: ${best_cc['strike']:.2f} (105-110% of current price)
        Expiration: {best_cc['days_to_exp']} days
        Income: ${best_cc['total_income']:,.2f}
        
        Benefits:
        • Generates immediate income
        • Provides {best_cc['annual_yield']:.1%} annualized yield
        • Reduces cost basis
        • Can be repeated monthly/quarterly
        
        Ideal for: Investors who believe stock will trade sideways
        or have modest gains, and want to generate income.
        """
    },
    'Balanced': {
        'primary': 'Collar',
        'details': f"""
        ✅ RECOMMENDED: Collar Strategy
        
        Put Strike: ${best_collar['put_strike']:.2f}
        Call Strike: ${best_collar['call_strike']:.2f}
        Expiration: {best_collar['days_to_exp']} days
        Net Cost: ${best_collar['total_net_cost']:,.2f}
        
        Benefits:
        • Low-cost or zero-cost protection
        • Defined risk range
        • Protects against {best_collar['protection_level']:.1%} downside
        • Allows for {best_collar['cap_level']:.1%} upside
        
        Ideal for: Investors who want downside protection 
        without paying large premiums, willing to cap upside.
        """
    },
    'Opportunistic': {
        'primary': 'Dynamic Strategy',
        'details': f"""
        ✅ RECOMMENDED: Dynamic Hedging Approach
        
        Current Market Conditions:
        • Volatility: {annual_volatility:.1%} ({"High" if annual_volatility > 0.4 else "Moderate" if annual_volatility > 0.25 else "Low"})
        • Technical Position: ${current_price:.2f}
        
        Suggested Approach:
        1. Start with 1/3 position in protective puts (1-2 month expiry)
        2. Sell covered calls on rallies (weekly/monthly)
        3. Adjust based on market conditions
        4. Consider collar during high volatility periods
        
        Benefits:
        • Flexibility to adapt to changing conditions
        • Optimize cost/benefit over time
        • Capture volatility premium
        
        Ideal for: Active investors comfortable with 
        options and willing to manage positions regularly.
        """
    }
}

print("\nBased on your 3,000 shares of Natera:\n")

for profile, rec in recommendations.items():
    print(f"\n🎯 {profile} Investor Profile:")
    print(rec['details'])

# Market outlook-based recommendations
print("\n" + "=" * 80)
print("MARKET OUTLOOK-BASED RECOMMENDATIONS")
print("=" * 80)

outlooks = {
    'Bullish': {
        'strategy': 'Covered Calls on rallies',
        'rationale': 'Generate income while participating in upside',
        'specific': f'Sell {int(SHARES_OWNED/100)} calls at ${current_price * 1.10:.2f}-${current_price * 1.15:.2f}'
    },
    'Bearish': {
        'strategy': 'Protective Puts',
        'rationale': 'Strong downside protection is priority',
        'specific': f'Buy {int(SHARES_OWNED/100)} puts at ${current_price * 0.90:.2f}-${current_price * 0.95:.2f}'
    },
    'Neutral': {
        'strategy': 'Collar or Iron Condor',
        'rationale': 'Profit from range-bound movement',
        'specific': f'Collar: Put ${current_price * 0.90:.2f} / Call ${current_price * 1.10:.2f}'
    },
    'Uncertain': {
        'strategy': 'Staggered Protection',
        'rationale': 'Diversify expiration dates and strikes',
        'specific': 'Split position: 1/3 each in 30, 60, 90-day protection'
    }
}

for outlook, details in outlooks.items():
    print(f"\n📈 {outlook} Market Outlook:")
    print(f"   Strategy: {details['strategy']}")
    print(f"   Rationale: {details['rationale']}")
    print(f"   Specific Action: {details['specific']}")

# Cost-benefit analysis
print("\n" + "=" * 80)
print("COST-BENEFIT ANALYSIS")
print("=" * 80)

position_value = current_price * SHARES_OWNED
print(f"\nYour Position: {SHARES_OWNED} shares × ${current_price:.2f} = ${position_value:,.2f}")

hedging_costs = {
    'No Hedge': {
        'Annual Cost': 0,
        'Protection': 'None',
        'Risk': f'${position_value:,.2f}'
    },
    'Protective Put': {
        'Annual Cost': best_pp['total_cost'] * (365 / best_pp['days_to_exp']),
        'Protection': f"{best_pp['protection_level']:.1%}",
        'Risk': f"${best_pp['max_loss']:,.2f}"
    },
    'Covered Call': {
        'Annual Cost': -best_cc['total_income'] * (365 / best_cc['days_to_exp']),
        'Protection': 'Income generation',
        'Risk': 'Opportunity cost if stock rallies'
    },
    'Collar': {
        'Annual Cost': best_collar['total_net_cost'] * (365 / best_collar['days_to_exp']),
        'Protection': f"{best_collar['protection_level']:.1%}",
        'Risk': f"${best_collar['max_loss']:,.2f}"
    }
}

print("\nAnnualized Hedging Costs:")
for strategy, metrics in hedging_costs.items():
    cost_pct = (metrics['Annual Cost'] / position_value) * 100 if position_value > 0 else 0
    if metrics['Annual Cost'] < 0:
        print(f"• {strategy}: Income of ${-metrics['Annual Cost']:,.0f}/year ({-cost_pct:.2f}% yield)")
    else:
        print(f"• {strategy}: Cost of ${metrics['Annual Cost']:,.0f}/year ({cost_pct:.2f}% of position)")
    print(f"    Protection: {metrics['Protection']}, Risk: {metrics['Risk']}")

## 8. Implementation Guide & Action Items

In [ ]:
# Generate actionable implementation guide
print("=" * 80)
print("IMPLEMENTATION GUIDE")
print("=" * 80)

print("""
📋 IMMEDIATE ACTION ITEMS:

1️⃣ ASSESS YOUR RISK TOLERANCE
   □ Determine maximum acceptable loss
   □ Identify investment time horizon
   □ Consider tax implications of strategies

2️⃣ CHOOSE YOUR STRATEGY
   Based on the analysis above, select one:
   □ Protective Put - Maximum protection, higher cost
   □ Covered Call - Income generation, capped upside
   □ Collar - Balanced protection, limited cost
   □ No hedge - Accept full risk/reward

3️⃣ EXECUTION CHECKLIST
   □ Check current bid-ask spreads
   □ Use limit orders (not market orders)
   □ Start with partial position (test with 1/3)
   □ Set calendar reminders for expiration dates
   □ Document strategy and rationale

4️⃣ POSITION SIZING GUIDELINES
""")

# Position sizing recommendations
contracts_needed = SHARES_OWNED / 100
print(f"\nYour position: {SHARES_OWNED} shares = {contracts_needed:.0f} option contracts")
print("\nRecommended sizing by strategy:")
print(f"• Protective Puts: Start with {int(contracts_needed/3)}-{int(contracts_needed/2)} contracts")
print(f"• Covered Calls: Start with {int(contracts_needed/3)}-{int(contracts_needed/2)} contracts")
print(f"• Collar: Implement on full position ({int(contracts_needed)} contracts)")

print("""
5️⃣ MONITORING & ADJUSTMENT
   Daily:
   □ Check position P&L
   □ Monitor stock price vs. strike prices
   
   Weekly:
   □ Review implied volatility changes
   □ Assess need for rolling positions
   
   At 21 Days to Expiration:
   □ Decide on rolling strategy
   □ Close or adjust positions
   
   At Expiration:
   □ Handle assignment/exercise
   □ Implement next cycle strategy

6️⃣ RISK MANAGEMENT RULES
   • Never let options expire worthless if they have value
   • Roll protective puts before expiration to maintain coverage
   • Consider closing covered calls if stock drops significantly
   • Adjust collar width based on volatility changes

7️⃣ TAX CONSIDERATIONS
   • Short-term vs. long-term capital gains
   • Wash sale rules for losses
   • Qualified covered calls for dividend treatment
   • Consult tax advisor for your specific situation
""")

# Generate specific trade orders
print("\n" + "=" * 80)
print("SAMPLE TRADE ORDERS")
print("=" * 80)

print(f"""
Based on current analysis, here are sample orders:

🔵 PROTECTIVE PUT ORDER:
   Action: BUY TO OPEN
   Quantity: {int(contracts_needed)} contracts
   Strike: ${best_pp['strike']:.2f}
   Expiration: {best_pp['days_to_exp']} days out
   Order Type: Limit Order
   Limit Price: ${best_pp['premium_per_share']:.2f} or better

🟢 COVERED CALL ORDER:
   Action: SELL TO OPEN
   Quantity: {int(contracts_needed)} contracts
   Strike: ${best_cc['strike']:.2f}
   Expiration: {best_cc['days_to_exp']} days out
   Order Type: Limit Order
   Limit Price: ${best_cc['premium_per_share']:.2f} or better

🔶 COLLAR ORDERS (execute simultaneously):
   Put - BUY TO OPEN:
      Quantity: {int(contracts_needed)} contracts
      Strike: ${best_collar['put_strike']:.2f}
      
   Call - SELL TO OPEN:
      Quantity: {int(contracts_needed)} contracts
      Strike: ${best_collar['call_strike']:.2f}
      
   Net Debit/Credit: ${best_collar['net_cost_per_share']:.2f}

⚠️ IMPORTANT REMINDERS:
• Verify all orders before submission
• Use limit orders during volatile markets
• Consider splitting large orders
• Keep records for tax purposes
""")

## 9. Executive Summary

### Key Findings
- Your 3,000 shares of Natera represent significant exposure that can benefit from strategic hedging
- Historical volatility analysis shows the stock has substantial price movement potential
- Multiple hedging strategies are available with different risk-reward profiles

### Top Recommendations
1. **For Maximum Protection**: Protective Put strategy provides downside insurance while maintaining upside
2. **For Income Generation**: Covered Call strategy generates premium income in sideways markets
3. **For Balanced Approach**: Collar strategy offers cost-effective protection with defined outcomes

### Next Steps
1. Review the analysis and determine your risk tolerance
2. Select appropriate strategy based on market outlook
3. Execute trades using the provided order templates
4. Monitor and adjust positions as needed

---
*This analysis is for educational purposes. Always consult with a financial advisor before implementing options strategies.*